# Superstore Sales Dashboard
**Goal:** Analyse 8,399 retail transactions to surface top-performing regions and products, identify underperforming segments, and understand what drives profit.

**Dataset:** Sample Superstore — Kaggle (2009–2012, Canada)  
**Sections:** Regional Performance · Category Analysis · Monthly Trends · Top Products · Underperformers · Discount Impact · Customer Segments

---
## Part 1 — Load & Overview

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Consistent colour palette
BLUE   = '#4e79a7'
ORANGE = '#f28e2b'
RED    = '#e15759'
GREEN  = '#59a14f'
PURPLE = '#b07aa1'

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 10})

df = pd.read_csv('Superstore-Sales.csv', encoding='latin-1')
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Year']  = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.to_period('M')
df['Profit Margin %'] = (df['Profit'] / df['Sales'] * 100).round(2)

print(f"Rows: {len(df):,}  |  Columns: {df.shape[1]}")
print(f"Date range: {df['Order Date'].min().date()} to {df['Order Date'].max().date()}")
print(f"Total Sales: ${df['Sales'].sum():,.0f}  |  Total Profit: ${df['Profit'].sum():,.0f}")
print(f"Overall Profit Margin: {df['Profit'].sum()/df['Sales'].sum():.1%}")
df.head(3)

---
## Part 2 — Regional Performance
Sales and profit broken down by region — to identify which regions generate revenue vs which generate profit.

In [ ]:
regional = df.groupby('Region')[['Sales', 'Profit']].sum().sort_values('Sales', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

regional['Sales'].plot(kind='barh', ax=axes[0], color=BLUE, edgecolor='white')
axes[0].set_title('Total Sales by Region', fontweight='bold')
axes[0].set_xlabel('Sales ($)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

colors = [RED if v < 0 else GREEN for v in regional['Profit']]
regional['Profit'].plot(kind='barh', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_title('Total Profit by Region', fontweight='bold')
axes[1].set_xlabel('Profit ($)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
axes[1].axvline(0, color='black', linewidth=0.8, linestyle='--')

plt.suptitle('Regional Performance — Sales vs Profit', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop region by Sales :", regional['Sales'].idxmax(),  f"(${regional['Sales'].max():,.0f})")
print("Top region by Profit:", regional['Profit'].idxmax(), f"(${regional['Profit'].max():,.0f})")
print("\nRegional summary:")
print(regional.sort_values('Sales', ascending=False).to_string())

---
## Part 3 — Category & Sub-Category Analysis
Revenue volume does not equal profitability — this section compares both.

In [ ]:
cat = df.groupby('Product Category')[['Sales', 'Profit']].sum()
cat['Margin %'] = (cat['Profit'] / cat['Sales'] * 100).round(1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

cat['Sales'].sort_values().plot(kind='barh', ax=axes[0], color=BLUE, edgecolor='white')
axes[0].set_title('Sales by Category', fontweight='bold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

cat['Profit'].sort_values().plot(kind='barh', ax=axes[1], color=GREEN, edgecolor='white')
axes[1].set_title('Profit by Category', fontweight='bold')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

cat['Margin %'].sort_values().plot(kind='barh', ax=axes[2], color=PURPLE, edgecolor='white')
axes[2].set_title('Profit Margin % by Category', fontweight='bold')
axes[2].set_xlabel('Margin (%)')

plt.suptitle('Category Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nCategory summary:")
print(cat.to_string())

---
## Part 4 — Monthly Sales Trend
Year-on-year monthly revenue to surface seasonality.

In [ ]:
monthly = df.groupby('Month')['Sales'].sum()

fig, ax = plt.subplots(figsize=(13, 4))
monthly.plot(ax=ax, color=BLUE, linewidth=1.8)
ax.fill_between(range(len(monthly)), monthly.values, alpha=0.15, color=BLUE)
ax.set_title('Monthly Sales Trend (2009–2012)', fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Sales ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
# Mark Q4 of each year
for idx, (period, val) in enumerate(monthly.items()):
    if str(period).endswith(('-10', '-11', '-12')):
        ax.axvspan(idx - 0.5, idx + 0.5, color=ORANGE, alpha=0.08)
ax.annotate('Q4 spikes (shaded)', xy=(0.82, 0.88), xycoords='axes fraction',
            color=ORANGE, fontsize=9)
plt.tight_layout()
plt.show()

peak = monthly.idxmax()
print(f"Peak month: {peak} — ${monthly[peak]:,.0f}")

---
## Part 5 — Top 10 Products by Sales

In [ ]:
top10 = df.groupby('Product Name')[['Sales', 'Profit']].sum().nlargest(10, 'Sales')

fig, ax = plt.subplots(figsize=(10, 6))
colors = [GREEN if p >= 0 else RED for p in top10['Profit']]
top10['Sales'].sort_values().plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.set_title('Top 10 Products by Sales (green = profitable, red = loss-making)', fontweight='bold')
ax.set_xlabel('Total Sales ($)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
plt.tight_layout()
plt.show()

print(top10[['Sales', 'Profit']].sort_values('Sales', ascending=False).to_string())

---
## Part 6 — Underperforming Sub-Categories
Sub-categories with the lowest profit — including those generating net losses.

In [ ]:
subcat = df.groupby('Product Sub-Category')[['Sales', 'Profit']].sum()
subcat['Margin %'] = (subcat['Profit'] / subcat['Sales'] * 100).round(1)
bottom10 = subcat.nsmallest(10, 'Profit')

fig, ax = plt.subplots(figsize=(10, 5))
colors = [RED if v < 0 else ORANGE for v in bottom10['Profit']]
bottom10['Profit'].sort_values().plot(kind='barh', ax=ax, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Bottom 10 Sub-Categories by Profit (red = net loss)', fontweight='bold')
ax.set_xlabel('Total Profit ($)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

losses = subcat[subcat['Profit'] < 0].sort_values('Profit')
if not losses.empty:
    print("Sub-categories running at a NET LOSS:")
    print(losses[['Sales', 'Profit', 'Margin %']].to_string())
else:
    print("No sub-categories with net losses.")
print("\nAll sub-categories ranked by profit:")
print(subcat.sort_values('Profit').to_string())

---
## Part 7 — Discount Impact on Profit
Does heavy discounting hurt profitability? This scatter shows the relationship between discount rate and profit per order.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors_scatter = [GREEN if p >= 0 else RED for p in df['Profit']]
ax.scatter(df['Discount'], df['Profit'], alpha=0.15, s=8, c=colors_scatter)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Discount Rate vs Profit per Order', fontweight='bold')
ax.set_xlabel('Discount Applied')
ax.set_ylabel('Profit ($)')
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
plt.tight_layout()
plt.show()

# Binned summary
df['Discount Band'] = pd.cut(df['Discount'],
    bins=[-0.01, 0, 0.1, 0.2, 0.3, 0.5, 1.0],
    labels=['No discount', '1-10%', '11-20%', '21-30%', '31-50%', '51%+'])
disc_summary = df.groupby('Discount Band', observed=True)['Profit'].mean().round(2)
print("Average profit by discount band:")
print(disc_summary.to_string())

---
## Part 8 — Customer Segment Analysis

In [ ]:
seg = df.groupby('Customer Segment')[['Sales', 'Profit']].sum()
seg['Margin %'] = (seg['Profit'] / seg['Sales'] * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
seg['Sales'].sort_values().plot(kind='barh', ax=axes[0], color=BLUE, edgecolor='white')
axes[0].set_title('Sales by Customer Segment', fontweight='bold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

seg['Margin %'].sort_values().plot(kind='barh', ax=axes[1], color=PURPLE, edgecolor='white')
axes[1].set_title('Profit Margin % by Customer Segment', fontweight='bold')
axes[1].set_xlabel('Margin (%)')

plt.suptitle('Customer Segment Performance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(seg.sort_values('Sales', ascending=False).to_string())

---
## Summary — Key Business Insights

| Finding | Detail |
|---|---|
| **Top region** | West leads on Sales; compare Profit chart to see if this holds |
| **Lowest region** | Nunavut — 31x gap vs West; investigate distribution vs demand |
| **Q4 seasonality** | Sales spike every Q4 across all four years — plan inventory accordingly |
| **Discounting** | Orders with >30% discount frequently generate negative profit |
| **Underperformers** | Sub-categories with negative profit margins are candidates for repricing or discontinuation |
| **Category insight** | Office Supplies has highest order volume but may have thinner margins than Technology |

**Recommendation:** Discount policy is the most controllable lever — reducing deep discounts (>30%) on low-margin sub-categories would directly improve overall profitability without requiring new revenue.